# Exit research: a visual walk-through (NOT a trading recommendation)

**What this is:** a private, read-only visual explanation of the practice-data exit study. Charts show *hypothetical* alternatives under one-minute bid/ask assumptions, **not** actual broker fills, account balances or a proven new strategy. No OANDA, Cloud, or X requests are made. The live -25/+75 rule has not changed.

**Safety:** Keep this committed TEMPLATE unexecuted. Copy it to `notebooks/06_exit_comparison.local.ipynb` (ignored by Git) and open the **local copy** in Jupyter. Notebook outputs can contain sensitive hypothetical research results; do not commit, screenshot publicly, export or share the executed copy. See `notebooks/README.md`.

## The story so far — in everyday language

1. **Step 1 — Gather the receipts.** Download a limited, read-only window of practice trade receipts and minute-by-minute market prices. Match the trades to the price record; a few very fast/ambiguous exits cannot be reconstructed exactly. **No orders were sent.**
2. **Step 2 — Check our measuring tape.** Confirm that the bot's trade/no-trade decisions match the data, that the broker's profit/loss arithmetic is consistent with recorded fill prices and size, and that changing a backtest setting actually changes the backtest. A matching receipt is **not** proof we could have filled a different exit at that price.
3. **Step 3 — Try 'what if?' on paper.** Replay nine predeclared ways to close *the same hypothetical entries*, including the current rule, early full profits, partial profits, protected/trailing stops and an earlier time exit. Use cautious treatment of one-minute ordering and additional costs. **This does not pick a winner**: the comparable sample is tiny and the old multi-year source files are missing.

**How to read the charts:** Coverage shows which days can be compared. The colored grid shows daily differences from today's -25/+75 rule (on the same dates). Cost panels show what the assumed charges change. One-day bars explain how exits work. The final chart shows whether enough data exist even to draw descriptive uncertainty intervals.

## 0. Load ONLY the private, already-computed offline report

Run Jupyter from the repository root or from `notebooks/`. Edit `FOLDER_NAME` for a newer audited export. Run `python scripts/research/run_exit_study.py --folder START_END` first if the report does not exist. The notebook reads no `.env`, fills, account IDs or broker API. Select a cost scenario below; the default is the explicitly *modeled* adverse extra cost, not measured fees.

In [ ]:
from pathlib import Path
import json
import sys
import matplotlib.pyplot as plt
import yaml

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'research' / 'experiments.yml').is_file():
    raise FileNotFoundError('Start Jupyter from this repository root or its notebooks/ folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.research_visuals import (load_views, plot_coverage, plot_daily_differences,
                                  plot_cost_scenarios, plot_one_day, plot_evidence_gate)

FOLDER_NAME = '2026-08-20_2026-09-24'  # Change only to an existing, audited private export.
COST_CASE = 'adverse_extra_1pt_per_fill'  # Alternative: 'bid_ask_only' (no extra cost).
path = ROOT / 'data' / 'raw' / 'oanda_research' / FOLDER_NAME / 'exit_study.json'
if not path.is_file():
    raise FileNotFoundError('Private exit study missing. First run the OFFLINE scripts/research/run_exit_study.py with --folder START_END. See research/step3-exit-study.md.')
report = json.loads(path.read_text(encoding='utf-8'))
protocol = yaml.safe_load((ROOT / 'research' / 'experiments.yml').read_text(encoding='utf-8'))
views = load_views(report, protocol)  # Checks protocol, same-day pairing, assumptions and summary totals.
if COST_CASE not in views:
    raise ValueError('Select a declared cost case from research/experiments.yml.')
view = views[COST_CASE]
print('LOCAL research only | assumed costs:', COST_CASE)
print('Days exported:', len(view.coverage), '| days paired:', len(view.eligible_dates),
      '| current-rule trade days paired:', int(view.net_difference['baseline_trade'].sum()))
print('Evidence gate:', view.uncertainty[view.order[0]]['status'], '| no rule selected.')

## 1. First ask: are these days trustworthy enough to compare?

Each square is an exported New York session. Green = clean paired trade/no-trade day; orange = broker exit not definitively reconstructed; red = minute candle ordering could change a candidate exit. **A day excluded for any candidate is excluded for all candidates** in that cost case, so alternatives are compared fairly on the same days. Excluded days are not 'bad trades'—removing them can bias results.

In [ ]:
fig = plot_coverage(view)
plt.show()

## 2. Look day by day: where do rules differ from today's rule?

Rows follow the **predeclared** order, not a performance ranking. Each column is an identical eligible day across all nine rules. Blue = simulated net result better than today's rule on that day; red = worse; white = unchanged. These are **differences from a hypothetical baseline**, NOT actual profits, account balances or evidence of future returns. No-trade days are retained as zero differences.

In [ ]:
fig = plot_daily_differences(view)
plt.show()

## 3. Cost stress: what if every fill costs more?

Both panels keep the nine rules in the original order, rather than sorting by a short sample's result. One case counts observed bid/ask spread only; the other **assumes** an extra adverse point at entry and exit plus $2 per fill. A half-close adds another fill. Horizontal bars sum net differences **on each case's eligible dates**; if future cases have different eligible sets, do not compare their bar lengths directly. Even if a bar looks large, this is an exploratory arithmetic check, **not a winning strategy**.

In [ ]:
fig = plot_cost_scenarios(views)
plt.show()

## 4. A single example: what does each exit actually do?

This picks the *first clean trade day* automatically, not the most profitable day. Bars show simulated **gross index points per original unit**, before fees and extra slippage. The label beside each rule shows how it closed. For a partial exit, the bar averages both closing legs: half at +30 and half at -25 is only +2.5 gross points. Edit `EXAMPLE_DAY` to another date in `view.eligible_dates` if desired. Do not interpret a minute candle as proof a real partial order would fill.

In [ ]:
EXAMPLE_DAY = None  # None = first clean trade day; or use an ISO date from view.eligible_dates.
fig = plot_one_day(view, day=EXAMPLE_DAY)
plt.show()

## 5. Do we have enough evidence even to put uncertainty bands on a plot?

The predeclared minimum for descriptive paired week-block resampling is **26 distinct calendar weeks AND 100 current-rule trade days**. Below either threshold the analysis reports *insufficient history* and this notebook deliberately draws **no probability or confidence bands**. If a future audited sample passes, the displayed bands are still exploratory: old settings have been viewed before, nine rules are compared without a multiple-testing correction, and genuinely new paper trading is needed.

In [ ]:
fig = plot_evidence_gate(view)
plt.show()

## What we can say (and cannot say)

**Can say:** collected private practice evidence; checked the signal and recorded P&L arithmetic; built a reproducible, cautious offline comparison of nine exit ideas. We can visualize where the hypothetical outcomes differ.

**Cannot say:** which exit maximizes profits, that modeled partials can execute as assumed, or that a backtest is a live trading result. Original 2020–2024 raw CSVs are missing, tick ordering is unknown, and this sample is too short for credible policy selection. Obtain/audit longer historical bid/ask or tick data, then use genuinely new forward paper sessions before proposing any production change.